# 🎬 OpenSource Clipping Studio — Backend Server

Run the Web Studio backend on a Kaggle or Colab GPU and connect from the browser dashboard.

**Runtime:** Linux x86_64, Python 3.10–3.12 (prefer 3.12), Internet enabled.

Setup uses an isolated virtual environment and starts FastAPI in a fresh process. The public URL appears only after actual renderer imports and FFmpeg pass readiness checks. Keep the final cell running while using Studio.

See `docs/KAGGLE_STUDIO.md` in the repository for troubleshooting and dependency details.

## 1. Setup Project

Clone into a named directory without deleting existing files. Existing checkouts are preserved, not automatically updated. Ensure your checkout contains this notebook's matching `scripts/kaggle_studio.py` and requirements files before running setup.

In [ ]:
# Clone repository without deleting the notebook working directory.
import os
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/NaufalRizqullah/opensource-clipping.git"
workspace = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content") if Path("/content").is_dir() else Path.cwd()
PROJECT_DIR = Path.cwd() if (Path.cwd() / "scripts/kaggle_studio.py").is_file() else workspace / "opensource-clipping"
if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
elif not (PROJECT_DIR / ".git").is_dir():
    raise RuntimeError("Project directory already exists and is not a checkout. Select a different PROJECT_DIR.")
if not (PROJECT_DIR / "scripts/kaggle_studio.py").is_file():
    raise RuntimeError("Update this checkout to the revision containing the Kaggle Studio fixes, then rerun setup.")
os.chdir(PROJECT_DIR)
print("Project checkout ready. Existing outputs are retained.")

In [ ]:
# Fail fast and keep installer output visible. Do not pip-install into this kernel.
STUDIO_ENV = PROJECT_DIR / ".cache" / "kaggle-studio" / "venv"
STUDIO_PYTHON = STUDIO_ENV / "bin" / "python"
subprocess.run(
    [sys.executable, str(PROJECT_DIR / "scripts/kaggle_studio.py"), "setup", "--venv", str(STUDIO_ENV)],
    cwd=PROJECT_DIR, check=True,
)

In [ ]:
# Install FFmpeg only if missing; failed system commands stop this cell.
import shutil
if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True)
subprocess.run(["ffmpeg", "-version"], check=True, stdout=subprocess.DEVNULL)
print("FFmpeg ready.")

## 2. Configure API Keys

Add `NGROK_AUTHTOKEN` to Kaggle Secrets or Colab userdata. Configure `GOOGLE_API_KEY` for Gemini or `NVIDIA_API_KEY` for NVIDIA; `PEXELS_API_KEY` and `HF_TOKEN` support optional features.

Keys are passed through the environment only. Their values are never printed or written to `.env`. The managed server ignores existing `.env` files.

In [ ]:
import os

secret_readers = []
try:
    from kaggle_secrets import UserSecretsClient
    secret_readers.append(UserSecretsClient().get_secret)
except Exception:
    pass
try:
    from google.colab import userdata
    secret_readers.append(userdata.get)
except ImportError:
    pass

for name in ("GOOGLE_API_KEY", "NVIDIA_API_KEY", "PEXELS_API_KEY", "HF_TOKEN", "NGROK_AUTHTOKEN"):
    value = os.environ.get(name, "")
    for read_secret in secret_readers:
        if value:
            break
        try:
            value = read_secret(name) or ""
        except Exception:
            pass
    if value:
        os.environ[name] = value
    print(f"{name}: {'configured' if value else 'not configured'}")
del value


In [ ]:
# ──────────────────────────────────────────────────────────────────────
# YouTube Cookies Setup (Optional) - Solves yt-dlp 403 Forbidden errors
# ──────────────────────────────────────────────────────────────────────
import os
import shutil

COOKIES_PATH = os.path.abspath("youtube_cookies.txt")
cookie_candidates = [
    "/kaggle/input/datasets/muhammadnaufal/tiktok-secret-cookies/youtube_cookies.txt",
    "/content/youtube_cookies.txt",
    COOKIES_PATH,
]

source_cookie = next((p for p in cookie_candidates if os.path.isfile(p)), None)

if source_cookie:
    if os.path.realpath(source_cookie) != os.path.realpath(COOKIES_PATH):
        shutil.copy(source_cookie, COOKIES_PATH)
    print(f"✅ YouTube cookies aktif: {COOKIES_PATH}")
else:
    print("ℹ️ youtube_cookies.txt tidak ditemukan. Download akan dicoba tanpa cookies.")


## 3. Start Backend Server + Tunnel

This cell launches a fresh Python process from the isolated environment. Startup verifies actual renderer imports before opening ngrok; allow several minutes for the first check.

Copy the **Public URL** only after **BACKEND SERVER READY** appears, then paste it into Studio → Connect. Keep this cell running. Stop the cell or use Studio's Stop Server button to close the backend and tunnel.

If setup or readiness fails, fix that error and rerun; a healthy HTTP server alone is not sufficient.

In [ ]:
from scripts.kaggle_studio import runtime_environment, stop_process

# The supervisor and Uvicorn both use the isolated interpreter, never this kernel.
supervisor = subprocess.Popen(
    [str(STUDIO_PYTHON), str(PROJECT_DIR / "scripts/kaggle_studio.py"), "start", "--venv", str(STUDIO_ENV)],
    cwd=PROJECT_DIR, env=runtime_environment(STUDIO_ENV), start_new_session=True,
)
try:
    return_code = supervisor.wait()
    if return_code not in (0, 130):
        raise RuntimeError(f"Studio server exited with code {return_code}. Check the startup diagnostics above.")
except KeyboardInterrupt:
    print("Stopping Studio backend and tunnel...")
finally:
    stop_process(supervisor)
